# Phase 2 — Step 1: Fine-tune DistilBERT

Fine-tunes `distilbert-base-uncased` on `nikhilgbharadwaj/ai-reviews-augmented`
to detect AI-generated reviews.

**Runtime:** Google Colab with **L4 GPU**. Expected training time ~20 min.
**Output:** model pushed to Hugging Face Hub at `<your-username>/ai-review-detector-distilbert`.

### Before running
1. Runtime → Change runtime type → **L4 GPU**
2. Have your HF write token ready (https://huggingface.co/settings/tokens)

## 1. GPU check

In [ ]:
!nvidia-smi

## 2. Install dependencies

In [ ]:
!pip install -q -U transformers datasets accelerate huggingface_hub evaluate scikit-learn

## 3. Hugging Face login

Paste your **write** token when prompted.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 4. Config

Edit `HF_USERNAME` to match your HF account.

In [ ]:
import os
HF_USERNAME = "nikhilgbharadwaj"   # <-- change if needed
DATASET_REPO = f"{HF_USERNAME}/ai-reviews-augmented"
MODEL_REPO   = f"{HF_USERNAME}/ai-review-detector-distilbert"

BASE_MODEL = "distilbert-base-uncased"
MAX_LENGTH = 256       # reviews are short; 256 covers ~99% without waste
BATCH_SIZE = 64        # L4 has 24GB, safe at 256 seq len
LR = 2e-5
EPOCHS = 3
SEED = 42

import torch, random, numpy as np
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("CUDA:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0))

## 5. Load splits from your HF dataset

In [ ]:
from datasets import load_dataset

ds = load_dataset(DATASET_REPO)
print(ds)

# Sanity: which splits do we actually have?
print("\nSplits:", list(ds.keys()))
for k in ds:
    labels = ds[k]['label'] if 'label' in ds[k].column_names else []
    print(f"  {k:30s} n={len(ds[k]):>6}")

### Map split names

The upload step pushed parquet files. Depending on how HF auto-detected them,
splits may be named after the filenames. The next cell handles common cases.

In [ ]:
# Normalize split names
split_map = {}
for name in ds.keys():
    low = name.lower()
    if "train" in low: split_map["train"] = name
    elif "val" in low: split_map["validation"] = name
    elif "heldout" in low or "held_out" in low: split_map["heldout"] = name
    elif "test" in low: split_map["test"] = name

print("Detected:", split_map)
assert "train" in split_map and "validation" in split_map and "test" in split_map, \
    f"Missing required splits. Got: {list(ds.keys())}"

train_ds = ds[split_map["train"]]
val_ds   = ds[split_map["validation"]]
test_ds  = ds[split_map["test"]]
heldout_ds = ds[split_map["heldout"]] if "heldout" in split_map else None

print(f"train: {len(train_ds)}  val: {len(val_ds)}  test: {len(test_ds)}", \
      f"heldout: {len(heldout_ds) if heldout_ds else 'N/A'}")

## 6. Tokenize

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

# Cast label to int (parquet may store as int64 already, but be safe)
def cast_label(ex):
    ex["label"] = int(ex["label"])
    return ex

cols_to_remove = [c for c in train_ds.column_names if c not in ("text","label")]
train_tok = train_ds.map(cast_label).map(tokenize, batched=True, remove_columns=cols_to_remove)
val_tok   = val_ds.map(cast_label).map(tokenize, batched=True, remove_columns=cols_to_remove)
test_tok  = test_ds.map(cast_label).map(tokenize, batched=True, remove_columns=cols_to_remove)
heldout_tok = (heldout_ds.map(cast_label).map(tokenize, batched=True, remove_columns=cols_to_remove)
               if heldout_ds else None)

print("Tokenization done.")
print("Sample:", train_tok[0]["input_ids"][:20], "label:", train_tok[0]["label"])

## 7. Model + metrics

In [ ]:
from transformers import AutoModelForSequenceClassification
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=2,
    id2label={0: "human", 1: "ai"}, label2id={"human": 0, "ai": 1},
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
    preds = logits.argmax(-1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", zero_division=0)
    # FPR — critical for this project (false positives on humans destroy trust)
    tn = ((labels == 0) & (preds == 0)).sum()
    fp = ((labels == 0) & (preds == 1)).sum()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": p, "recall": r, "f1": f1, "fpr": float(fpr),
        "auc": roc_auc_score(labels, probs),
    }

## 8. Training

L4 supports bf16 — faster and more stable than fp16 for transformer training.

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

args = TrainingArguments(
    output_dir="./out",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LR,
    weight_decay=0.01,
    warmup_ratio=0.1,
    bf16=True,                      # L4 supports bf16 natively
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

## 9. Evaluate on test + held-out generator

The **held-out** number is the one that matters — it tells you whether the model learned AI-ness or just memorized generators it saw during training.

In [ ]:
import json

print("=" * 60)
print("VALIDATION (in-distribution)")
print("=" * 60)
val_metrics = trainer.evaluate(val_tok)
print(json.dumps(val_metrics, indent=2, default=str))

print("\n" + "=" * 60)
print("TEST (in-distribution)")
print("=" * 60)
test_metrics = trainer.evaluate(test_tok)
print(json.dumps(test_metrics, indent=2, default=str))

if heldout_tok is not None:
    print("\n" + "=" * 60)
    print("HELD-OUT GENERATOR  ← the honest number")
    print("=" * 60)
    heldout_metrics = trainer.evaluate(heldout_tok)
    print(json.dumps(heldout_metrics, indent=2, default=str))

### Reading these numbers

- **Test F1** in 0.92-0.97 range = healthy.
- **Held-out F1** within ~5-10 points of test F1 = good generalization.
- **Held-out F1** much lower (e.g., test 0.95, heldout 0.70) = model memorized
  the in-training generators. Still useful, but expect real-world accuracy
  closer to the held-out number.
- **FPR < 0.05** is the goal. Above 0.10 means too many real reviews flagged.

## 10. Save predictions (val set) — needed for the fusion layer later

In [ ]:
import pandas as pd

val_preds = trainer.predict(val_tok)
val_probs = torch.softmax(torch.tensor(val_preds.predictions), dim=-1).numpy()[:, 1]

val_df = pd.DataFrame({
    "text": val_ds["text"],
    "label": val_ds["label"],
    "distilbert_prob": val_probs,
})
val_df.to_parquet("val_distilbert_preds.parquet")

# Same for test, so we can score the fusion layer cleanly later
test_preds = trainer.predict(test_tok)
test_probs = torch.softmax(torch.tensor(test_preds.predictions), dim=-1).numpy()[:, 1]
test_df = pd.DataFrame({
    "text": test_ds["text"],
    "label": test_ds["label"],
    "distilbert_prob": test_probs,
})
test_df.to_parquet("test_distilbert_preds.parquet")

print("Saved val_distilbert_preds.parquet and test_distilbert_preds.parquet")
val_df.head()

## 11. Push model to Hugging Face Hub

So Phase 4 (the FastAPI on HF Spaces) can pull it.

In [ ]:
trainer.push_to_hub(MODEL_REPO, private=False)
tokenizer.push_to_hub(MODEL_REPO)
print(f"Model pushed: https://huggingface.co/{MODEL_REPO}")

## 12. Quick smoke test

In [ ]:
from transformers import pipeline
clf = pipeline("text-classification", model=trainer.model, tokenizer=tokenizer,
               device=0, top_k=None)

samples = [
    "Best purchase ever! High quality and amazing value. Would buy again 5/5 stars.",
    "Got this last Tuesday. The latch is a bit stiff but works fine after a week. My cat ignored it for two days then claimed it.",
    "5/5 stars! As a busy parent, I was impressed by the sleek design and intuitive features of this product.",
    "broken on arrival. seller refunded fast tho",
]
for s in samples:
    out = clf(s)[0]
    ai = next(x for x in out if x["label"] == "ai")
    print(f"AI={ai['score']:.3f}  |  {s[:80]}")

## Done — what to do next

You now have:
- A fine-tuned model on HF Hub
- `val_distilbert_preds.parquet` and `test_distilbert_preds.parquet` saved (download these — Phase 2 step 4 fusion needs them)

**Next:** Phase 2 step 2 = stylometric features. We'll compute perplexity, burstiness, type-token ratio, etc. on the same val and test sets, then merge with the DistilBERT probs and train the XGBoost fusion layer.

Download both parquet files from the Colab file browser before closing the runtime.